## Import Libraries and Data

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import sys

In [ ]:
os.system(f'git clone https://github.com/m-j-r-q/Cura.git /kaggle/working/Cura')
sys.path.append('/kaggle/working/Cura/backend/src')

IMAGE_DIR  = '/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224'
DATA_DIR = '/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized'
CSV_PATH  = f'{DATA_DIR}/Data_Entry_2017.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("Session ready.")

Cloning into '/kaggle/working/Cura'...


Device: cpu
Session ready.


Updating files: 100% (24/24), done.


In [3]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows.")

Loaded 112120 rows.


## Create Label Vector: DISEASES

In [4]:
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pleural_Thickening',
    'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Hernia'
]

NUM_CLASSES = len(DISEASES)
print(f"{NUM_CLASSES} disease classes")

14 disease classes


## Function For Encoding Labels

In [5]:
def encode_labels(label_string):
    vector = torch.zeros(NUM_CLASSES)
    if label_string == 'No Finding':
        return vector
    for disease in label_string.split('|'):
        disease = disease.strip()
        if disease in DISEASES:
            idx = DISEASES.index(disease)
            vector[idx] = 1.0
    return vector

In [7]:
test = encode_labels("Cardiomegaly|Emphysema")
print(test)
print(test.sum().item(), "diseases flagged")

tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
2.0 diseases flagged


## Dataset Class with Member Functions Required by Pytorch.

In [8]:
class CuraDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(self.image_dir, row['Image Index'])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label_vector = encode_labels(row['Finding Labels'])

        return image, label_vector

## Compute Mean and Standard Deviation of Pixel Channels for Normalization

In [9]:
from torch.utils.data import DataLoader

sample_df = df.sample(5000, random_state=42)

stat_transform = transforms.Compose([
    transforms.ToTensor()
])

stat_dataset = CuraDataset(sample_df, IMAGE_DIR, transform=stat_transform)
stat_loader = DataLoader(stat_dataset, batch_size=32, num_workers=0)

channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
total_pixels = 0

for images, _ in stat_loader:
    batch_size = images.size(0)

    images = images.view(batch_size, 3, -1)

    pixels_in_batch = batch_size * images.size(2)
    total_pixels += pixels_in_batch

    channel_sum += images.sum(dim=[0, 2])
    channel_squared_sum += (images ** 2).sum(dim=[0, 2])


mean = channel_sum / total_pixels
std = torch.sqrt((channel_squared_sum / total_pixels) - (mean ** 2))
print(f"Dataset mean: {mean}")
print(f"Dataset std:  {std}")

Dataset mean: tensor([0.4967, 0.4967, 0.4967])
Dataset std:  tensor([0.2478, 0.2478, 0.2478])


## Pytorch Transforms

In [10]:
CURA_MEAN = [0.4967, 0.4967, 0.4967]
CURA_STD  = [0.2478, 0.2478, 0.2478]

train_transform = transforms.Compose([
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CURA_MEAN, CURA_STD)
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CURA_MEAN, CURA_STD)
])

## Training, Validation and Test Split

Ordered according to PatientID rather than image, such that no unique patient is present in more than one split.


In [11]:
patient_ids = df['Patient ID'].unique()
print(f"Total unique patients: {len(patient_ids)}")

np.random.seed(42)
np.random.shuffle(patient_ids)

n = len(patient_ids)
train_ids = set(patient_ids[:int(0.8*n)])
val_ids   = set(patient_ids[int(0.8*n):int(0.9*n)])
test_ids  = set(patient_ids[int(0.9*n):])

train_df = df[df['Patient ID'].isin(train_ids)].reset_index(drop=True)
val_df   = df[df['Patient ID'].isin(val_ids)].reset_index(drop=True)
test_df  = df[df['Patient ID'].isin(test_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} images")
print(f"Val:   {len(val_df)} images")
print(f"Test:  {len(test_df)} images")

Total unique patients: 30805
Train: 89703 images
Val:   11221 images
Test:  11196 images


In [12]:
train_patients = set(train_df['Patient ID'])
val_patients   = set(val_df['Patient ID'])
test_patients  = set(test_df['Patient ID'])

assert len(train_patients & val_patients) == 0, "Leakage: train/val overlap"
assert len(train_patients & test_patients) == 0, "Leakage: train/test overlap"
assert len(val_patients & test_patients) == 0, "Leakage: val/test overlap"

print("No leakage detected.")
print(f"Train patients: {len(train_patients)}")
print(f"Val patients:   {len(val_patients)}")
print(f"Test patients:  {len(test_patients)}")

No leakage detected.
Train patients: 24644
Val patients:   3080
Test patients:  3081


## Data Loader

In [13]:
train_dataset = CuraDataset(train_df, IMAGE_DIR, transform=train_transform)
val_dataset   = CuraDataset(val_df,   IMAGE_DIR, transform=val_transform)
test_dataset  = CuraDataset(test_df,  IMAGE_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 2804
Val batches:   351
Test batches:  350


## Compute Class Weights

In [14]:
pos_counts = torch.zeros(NUM_CLASSES)
neg_counts = torch.zeros(NUM_CLASSES)

for label_string in train_df['Finding Labels']:
    vector = encode_labels(label_string)
    pos_counts += vector
    neg_counts += (1 - vector)

pos_weight = neg_counts / pos_counts

print("Disease weights (higher = rarer disease):")
for i, disease in enumerate(DISEASES):
    print(f"{disease:25} pos={int(pos_counts[i]):5}  weight={pos_weight[i]:.1f}x")

Disease weights (higher = rarer disease):
Atelectasis               pos= 9170  weight=8.8x
Cardiomegaly              pos= 2203  weight=39.7x
Effusion                  pos=10506  weight=7.5x
Infiltration              pos=16022  weight=4.6x
Mass                      pos= 4499  weight=18.9x
Nodule                    pos= 5045  weight=16.8x
Pneumonia                 pos= 1098  weight=80.7x
Pleural_Thickening        pos= 2667  weight=32.6x
Pneumothorax              pos= 4357  weight=19.6x
Consolidation             pos= 3717  weight=23.1x
Edema                     pos= 1816  weight=48.4x
Emphysema                 pos= 2068  weight=42.4x
Fibrosis                  pos= 1392  weight=63.4x
Hernia                    pos=  192  weight=466.2x


## Pipeline Check

In [15]:
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"Image dtype: {images.dtype}")
print(f"Label dtype: {labels.dtype}")
print(f"Pixel value range: [{images.min():.3f}, {images.max():.3f}]")
print(f"Labels in first sample: {labels[0]}")
print(f"Diseases present: {[DISEASES[i] for i in range(NUM_CLASSES) if labels[0][i] == 1]}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 14])
Image dtype: torch.float32
Label dtype: torch.float32
Pixel value range: [-2.004, 2.031]
Labels in first sample: tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])
Diseases present: ['Pleural_Thickening']
